# Representations and Costs

```{contents}
:local:
:depth: 2
```

Once an ADT contract is clear, the next question is representation: how should the data be stored so the required operations are correct and efficient?


```{index} One Behavior Can Have Different Physical Work
```



```{index} One Behavior Can Have Different Physical Work
```

## One Behavior Can Have Different Physical Work



Use the undo contract from the preceding lesson. An array representation might keep active entries in positions zero through Count minus one. A linked representation might keep the newest entry at the head. The same Push/Pop history can be correct in both, yet their storage and work differ. To compare them fairly, hold the abstract behavior fixed and change only the representation.

For the array, Count and capacity answer different questions. Count describes active entries; capacity describes available slots in the allocated backing storage. A push with unused capacity writes one entry and increases Count. A push into full storage may allocate a larger array, copy active entries, and then append. Calling both pushes constant work ignores the copying performed by the growth case.

For a linked representation, a push can create one node whose next reference identifies the old head. Updating the head then makes the new node first. A pop can advance head to its next node. This avoids an array-wide growth copy, but node allocation and extra references are still real costs. Neither representation is universally better merely because one operation has a simple asymptotic label.

| Representation question | Array-backed history | Linked history |
| --- | --- | --- |
| How are active values located? | Used prefix of slots | Nodes reachable from head |
| What marks empty? | Count equals zero | No head node, with matching Count |
| What may make a push expensive? | Growth and copying | Allocation and runtime costs |
| What must stay hidden? | Unused capacity | Internal node links |

For related explanation, see the further reading below.[^development-reading]


```{index} Define What You Count
```



```{index} Define What You Count
```

## Define What You Count



If the input size is the number n of active entries, counting element copies explains an array growth. Counting only calls to Push hides the difference among calls. Big-O summarizes a bound as n grows; it does not supply the cost model for you. State whether the analysis counts comparisons, copies, reference updates, or another operation.

An amortized append claim concerns a whole sequence of operations, including occasional expensive growth. It is not a claim that every individual append is cheap, and it is not an average over randomly selected inputs. Likewise, constant auxiliary storage describes additional working memory under a model; it does not mean the data structure itself occupies no memory.

Guided comparison: an array with Count three and capacity four receives two pushes. Explain why the first can use a free slot and the second may require growth. Draw the abstract history after both calls, then draw the different physical storage states. Keeping those drawings separate prevents a copied backing array from being mistaken for a changed ADT result.


```{index} representation; data structure
```
```{index} representation independence
```



```{index} One Contract, Multiple Representations
```

## One Contract, Multiple Representations



The same stack ADT can be represented in more than one way:

| Representation | Main idea |
| --- | --- |
| Array-backed stack | Store items in contiguous slots and track the top index. |
| List-backed stack | Delegate storage growth to `List<T>`. |
| Linked-node stack | Store each item in a node that references the next node. |

If all three preserve the same contract, client code can use them through the same interface. This is **representation independence**.

What changes between the three rows is private: the fields, the way `Push` finds the next free place, and the way `Pop` locates the most recent item. What does not change is everything a client can observe: the sequence of returned values, the `Count` after each call, and the empty-stack policy. That is why a client variable should be declared with the interface type. Changing `IStack<string> history = new ListStack<string>();` to `new LinkedStack<string>()` is a one-token edit at the point of construction, and no other line needs to know. Chapter 14 develops the array and linked representations in full, including growth, middle insertion, and their invariants; this section introduces each just far enough to compare their costs.

```{index} array-backed stack
```



```{index} Array-Backed Stack
```

## Array-Backed Stack



An array-backed stack keeps its items in a fixed-size array and tracks how many slots are in use. The representation has two fields and one invariant that ties them together:

```csharp
private T[] items;          // capacity: items.Length slots are allocated
public int Count { get; private set; }   // the first Count slots hold the stack, bottom to top
// Invariant: items[0 .. Count-1] are the stack's values; items[Count ..] are unused.
```

The top of the stack is always `items[Count - 1]`, and the next free slot is always `items[Count]`, so neither operation has to search. The diagram shows a stack of three in a five-slot array:

```text
items: [A] [B] [C] [ ] [ ]
count: 3
top:   C
```

Pushing has two steps, and a third when the array is full: if `Count == items.Length`, allocate a larger array and copy the used slots into it; then write the item into `items[Count]`; then increment `Count`. Popping reverses the order: decrement `Count`, read `items[Count]`, and clear that slot so the array does not keep a reference to a value the stack no longer owns. The example below implements exactly these steps with a starting capacity of four and doubles the array when it fills.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
var stack = new ArrayStack<int>();
stack.Push(10);
stack.Push(20);
Console.WriteLine(stack.Pop());
Console.WriteLine(stack.Count);

class ArrayStack<T>
{
    private T[] items = new T[4];
    public int Count { get; private set; }

    public void Push(T item)
    {
        if (Count == items.Length) Array.Resize(ref items, items.Length * 2);
        items[Count] = item;
        Count++;
    }

    public T Pop()
    {
        if (Count == 0) throw new InvalidOperationException("Stack is empty.");
        Count--;
        T value = items[Count];
        items[Count] = default!;
        return value;
    }
}

20
1


The output is `20` and then `1`. After two pushes the used prefix was `[10, 20]` with `Count` 2; `Pop` decremented `Count` to 1 and returned `items[1]`, which was 20. The value 10 is still the top. Nothing was resized, because four slots were enough; a fifth push would trigger `Array.Resize`, which allocates an eight-slot array, copies the four values, and makes `items` refer to the new array. That copy is the occasional O(n) cost the next subsection accounts for.

Two mistakes are common in this code. Reading `items[Count]` in `Pop` before decrementing returns the unused slot, not the top, and a stack that does that will appear to pop default values. Omitting `items[Count] = default!` does not break the contract, but it keeps a reference alive in the unused slot, so a large object that the client has already discarded cannot be reclaimed until the slot is overwritten.

```{index} linked nodes
```
```{index} reference; linked structure
```



```{index} Linked Nodes and References
```

## Linked Nodes and References



A linked representation stores each item in a small object called a **node**. Each node holds one value and a reference to the next node; the last node's reference is `null`. The stack itself is just one reference to the top node:

```csharp
class Node<T>
{
    public T Value { get; }
    public Node<T>? Next { get; }      // null marks the end of the chain
    public Node(T value, Node<T>? next) { Value = value; Next = next; }
}
Node<string>? top = null;              // an empty stack is a null top
```

```text
top -> [C | next] -> [B | next] -> [A | null]
```

The variable `top` does not contain the whole stack. It contains a reference to the first node, and following `Next` references reveals the rest. Pushing creates one node whose `Next` is the current top and then makes `top` refer to the new node; popping makes `top` refer to `top.Next`. No slot is searched and no array is copied, so each operation does a fixed amount of work regardless of how many nodes exist. The example below pushes A, B, and C by prepending, then walks the chain from the top.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
Node<string>? top = null;
top = new Node<string>("A", top);
top = new Node<string>("B", top);
top = new Node<string>("C", top);

for (Node<string>? current = top; current is not null; current = current.Next)
{
    Console.WriteLine(current.Value);
}

class Node<T>
{
    public T Value { get; }
    public Node<T>? Next { get; }

    public Node(T value, Node<T>? next)
    {
        Value = value;
        Next = next;
    }
}

C
B
A


The output is `C`, `B`, `A`: the most recently pushed value is at the head of the chain, which is exactly the stack order. Each `new Node<string>(value, top)` captured the old top as its `Next` before the assignment replaced `top`, so the chain grew at the front. The `for` loop is the standard traversal pattern for linked structures: start at the head, stop when the current reference is `null`, and advance with `current = current.Next`.

The order of the two steps in a push matters. Writing `top = new Node<string>("D", null)` would attach D to nothing and make the earlier three nodes unreachable; the old chain is lost without any error message. The same care applies to popping: read `top.Value` before `top = top.Next`, or the value is gone. Chapter 14 traces these reference updates in detail.

```{index} Counting Work and Big-O
```

## Counting Work and Big-O




```{index} Big O; input size; auxiliary space
```


Let `n` be the number of records. Count a representative operation before comparing implementations. **Big-O** describes an upper bound on how work grows with input size, ignoring fixed factors for sufficiently large inputs. It is not an elapsed time or automatically a worst-case claim.

| Bound | Example | Effect of doubling input |
|---|---|---|
| O(1) | Reading one array slot | Bounded work independent of size. |
| O(log n) | Repeatedly halving a search interval | About one extra halving. |
| O(n) | Scanning all records | About twice the work. |
| O(n log n) | A typical comparison sort | Slightly more than twice the work. |
| O(n²) | Comparing every pair | About four times the work. |

These are useful upper bounds; a linear scan is also O(n²), but O(n) is more informative. A **tight bound**, written Θ, describes the actual growth order under the stated model. Later chapters use Θ when both upper and lower growth bounds are known.


In [1]:
using System.Collections.Generic;
using System;
using System.Linq;

foreach (int n in new[] { 0, 1, 4, 8 })
{
    int[] expenses = Enumerable.Repeat(125, n).ToArray();
    long total = 0, additions = 0;
    foreach (int cents in expenses) { total += cents; additions++; }
    Console.WriteLine($"n={n}, total={total}, additions={additions}");
}

n=0, total=0, additions=0
n=1, total=125, additions=1
n=4, total=500, additions=4
n=8, total=1000, additions=8


This example counts exactly `n` additions. Array access and fixed-width arithmetic are treated as constant-cost operations. A comparison of long strings can depend on string length.

Specify the case: a search may stop at the first record or inspect all `n`. **Amortized cost** averages over a sequence of operations: geometric array growth permits O(1) amortized append even though a resizing append takes O(n). This does not make every append constant-time. **Auxiliary space** counts extra storage beyond the input; a running total takes O(1), while an additional copy takes O(n). Chapter 21 develops formal bounds and richer cost models.

Before running, predict the addition counts for sizes zero, one, four, and eight. Then explain why reading only `expenses[0]` has a different growth bound and needs a nonempty-input precondition.


```{index} Best, Worst, and Average Cases
```

### Best, Worst, and Average Cases

A growth bound is incomplete until it says *which inputs* it describes. Consider a linear search for a customer ID in an unsorted array of n records. If the ID is in the first slot, one comparison settles it; if the ID is in the last slot or absent, every slot is compared. The same algorithm on inputs of the same size does very different amounts of work, so analysts name three cases.

| Case | Question it answers | Linear search of n records | When it is the right measure |
| --- | --- | --- | --- |
| Best | Least work any input of size n can cause | 1 comparison | Rarely useful alone; mainly shows what is possible |
| Worst | Most work any input of size n can cause | n comparisons | Deadlines, real-time systems, any guarantee |
| Average | Expected work over a stated distribution of inputs | About n/2 comparisons if every position is equally likely and the key is present | Throughput over many queries |

The worst case is the most common choice because it is a guarantee: no input, however unlucky, costs more. The average case needs an assumption about how inputs are distributed, and that assumption must be stated. If most searches are for recently added customers who sit at the end of the array, the uniform average of n/2 is wrong for that workload. The example counts comparisons for the three situations directly.

In [ ]:
using System;

string[] ids = { "C-104", "C-117", "C-121", "C-133", "C-140", "C-152", "C-166", "C-171" };

Report("C-104");   // best case: first slot
Report("C-171");   // worst case for a present key: last slot
Report("C-999");   // worst case overall: absent, every slot compared
double averagePresent = 0;
foreach (string id in ids) averagePresent += CountComparisons(id);
Console.WriteLine($"average over present keys: {averagePresent / ids.Length:F1} comparisons");

void Report(string key) => Console.WriteLine($"{key}: {CountComparisons(key)} comparisons");

int CountComparisons(string key)
{
    int comparisons = 0;
    for (int i = 0; i < ids.Length; i++)
    {
        comparisons++;
        if (ids[i] == key) return comparisons;
    }
    return comparisons;
}

C-104: 1 comparisons
C-171: 8 comparisons
C-999: 8 comparisons
average over present keys: 4.5 comparisons


The first search makes 1 comparison, the last-slot search makes 8, and the absent key also makes 8. The average over the eight present keys is 4.5, which is (n + 1) / 2 for n = 8: each position i costs i comparisons, and the mean of 1 through n is (n + 1) / 2. All three cases are O(n) in the bound sense, because the worst case grows linearly, but reporting only "O(n)" would hide that a successful search typically costs half of that. When you state a cost from now on, name the case: "linear search is n comparisons in the worst case and about n/2 on average when the key is present and every position is equally likely."

```{index} operation cost
```
```{index} Big O; collection operations
```



```{index} Operation Costs
```

## Operation Costs



A data structure is not just correct or incorrect. It also has costs.

| Operation | Array/List-backed stack | Linked-node stack |
| --- | --- | --- |
| `Push` | Usually O(1); occasional resize costs O(n) | O(1) |
| `Pop` | O(1) | O(1) |
| `Peek` | O(1) | O(1) |
| Random access by index | O(1), if exposed | O(n) |
| Extra memory per item | low | one extra reference per node |

The best representation depends on the workload. If a program mostly pushes and pops at one end, either representation can work. If it needs fast index access, linked nodes are the wrong fit.

Time is only half of the comparison. The two representations also spend memory differently, and the difference is predictable. An array-backed stack allocates `capacity` slots whether or not they are used, so its storage is `capacity × E`, where E is the size of one element. A linked stack allocates one node per stored item, and each node carries one extra reference of size P alongside its element, so its storage is `n × (E + P)` for n items. The array wastes the unused slots; the linked structure pays a reference per item. Setting the two expressions equal gives a break-even point: the array is more compact once

```text
n > capacity × E / (E + P)
```

When elements and references are the same size, that is `n > capacity / 2`: the array wins whenever it is more than half full, and the linked structure wins whenever it is emptier than that. The example evaluates both expressions for a stack whose capacity is 1,000 with 8-byte elements and 8-byte references.

In [ ]:
using System;

const int capacity = 1000;
const int E = 8;   // bytes per element (a long or a reference)
const int P = 8;   // bytes per next reference

Console.WriteLine($"break-even n = {capacity * E / (E + P)}");
Console.WriteLine("    n   array bytes  linked bytes  more compact");
foreach (int n in new[] { 100, 250, 500, 750, 1000 })
{
    int arrayBytes = capacity * E;
    int linkedBytes = n * (E + P);
    string winner = arrayBytes < linkedBytes ? "array" : arrayBytes > linkedBytes ? "linked" : "tie";
    Console.WriteLine($"{n,5}   {arrayBytes,11}  {linkedBytes,12}  {winner}");
}

break-even n = 500
    n   array bytes  linked bytes  more compact
  100          8000          1600  linked
  250          8000          4000  linked
  500          8000          8000  tie
  750          8000         12000  array
 1000          8000         16000  array


The array's cost is a flat 8,000 bytes at every row, because it was allocated for 1,000 slots regardless of use. The linked cost grows 16 bytes per item, crosses the array at n = 500, and is twice the array's at full occupancy. The rule of thumb follows directly: when the number of items varies widely or is unknown, a linked representation does not pay for slots it never fills; when the size is known in advance and the structure stays near full, the array is tighter by a constant factor and also keeps its elements adjacent in memory, which the hardware rewards. Neither conclusion depends on the asymptotic labels in the table above, both representations are O(n) in space; the decision lives in the constants.

```{index} Code Tracing: Array Stack State
```

### Code Tracing: Array Stack State

The `ArrayStack<int>` from earlier in this section starts with a four-slot array and doubles when full. Trace the program below by hand. After each statement, record `Count`, the array's length, and the contents of the used prefix. Then predict the three printed lines.

```csharp
var s = new ArrayStack<int>();
s.Push(5); s.Push(8); s.Push(13);
Console.WriteLine(s.Pop());
s.Push(21); s.Push(34); s.Push(55);
Console.WriteLine($"{s.Count} items, capacity {s.Capacity}");
Console.WriteLine(s.Pop() + s.Pop());
```

For this exercise the class exposes a read-only `Capacity` property that returns `items.Length`; it exists only so the trace can be checked.

In [ ]:
// Solution
using System;

var s = new ArrayStack<int>();
s.Push(5); s.Push(8); s.Push(13);          // Count 3, capacity 4, prefix [5, 8, 13]
Console.WriteLine(s.Pop());                // 13; Count 2
s.Push(21); s.Push(34);                    // Count 4, capacity 4, prefix [5, 8, 21, 34]
s.Push(55);                                // full: resize to 8, then Count 5, prefix [5, 8, 21, 34, 55]
Console.WriteLine($"{s.Count} items, capacity {s.Capacity}");
Console.WriteLine(s.Pop() + s.Pop());      // 55 + 34

class ArrayStack<T>
{
    private T[] items = new T[4];
    public int Count { get; private set; }
    public int Capacity => items.Length;

    public void Push(T item)
    {
        if (Count == items.Length) Array.Resize(ref items, items.Length * 2);
        items[Count] = item;
        Count++;
    }

    public T Pop()
    {
        if (Count == 0) throw new InvalidOperationException("Stack is empty.");
        Count--;
        T value = items[Count];
        items[Count] = default!;
        return value;
    }
}

13
5 items, capacity 8
89


The lines are `13`, `5 items, capacity 8`, and `89`. The resize happened on the push of 55, not of 34: after popping 13 the prefix had two items, two more pushes filled the four slots exactly, and only the sixth push found `Count == items.Length`. A trace that doubles the capacity on the push of 34 has forgotten that the pop freed a slot. The last line adds 55 and 34, the two most recent pushes, in that order.

```{index} Growth-Rate Ordering
```

### Growth-Rate Ordering

Arrange these seven cost functions from slowest growing to fastest growing as n becomes large, then open the answer. Two of them belong to the same growth class.

`5n`, `n log n`, `2ⁿ`, `n²`, `log n`, `1000`, `n / 2`

```{dropdown} Answer
From slowest to fastest: `1000` (constant), `log n`, then `n / 2` and `5n` together (both linear; the constant factors do not separate growth classes), then `n log n`, then `n²`, then `2ⁿ`.

Two checks help. Doubling n adds a fixed amount to `log n`, doubles any linear function, slightly more than doubles `n log n`, quadruples `n²`, and squares `2ⁿ`. And for small n the order can look different: `1000` exceeds `n²` until n passes 31, which is why asymptotic claims always say "for sufficiently large n."
```

```{index} Practice
```

## Practice



1. Draw the linked-node diagram after pushing `"red"`, `"blue"`, and `"green"` onto a stack.
2. Explain why resizing an array-backed stack is occasional O(n), even though most pushes are O(1).
3. Choose a representation for an undo history and justify it using operation costs.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. An array-backed stack reports O(1) amortized push. A teammate concludes that every push finishes in the same small time. What is wrong?

```{dropdown} Answer
Amortized cost is an average over a whole sequence. Most pushes are constant, but the push that triggers a resize copies every stored item and is O(n) on its own. The guarantee is about the total cost of n pushes, not about any single push.
```

2. Which case, best, worst, or average, does a service-level agreement that promises "every lookup under 50 milliseconds" require you to analyze?

```{dropdown} Answer
The worst case. A promise about every lookup can be kept only if the most expensive input still meets it; average-case figures describe throughput, not a guarantee.
```

3. A stack holds 120 items in an array allocated for 4,000, with 8-byte elements and 8-byte references. Which representation is using less memory right now?

```{dropdown} Answer
The linked one. The array holds 4,000 × 8 = 32,000 bytes regardless of use; the linked stack holds 120 × 16 = 1,920 bytes. The break-even point is 2,000 items, far above the current 120.
```

4. Why can a linked stack not offer O(1) access to the item at position k?

```{dropdown} Answer
The only reference it holds is to the top node, and each node knows only its successor. Reaching position k means following k references, which is O(k) work and O(n) in the worst case. An array computes the address of slot k directly.
```

```{index} Summary
```

## Summary



A representation is the hidden storage strategy behind an ADT contract. Arrays, lists, and linked nodes make different tradeoffs in speed, memory use, and flexibility. Data-structure design is the art of matching those tradeoffs to the operations a program performs most often.


```{rubric} Footnotes
```


[^1302_representations_costs-1]: Big-O compares growth under an explicit cost model. Chapter 21 extends these foundations with formal bounds, correctness arguments, and recurrences.


[^development-reading]: Further reading: [OpenDSA CS3, Chapter Introduction: Lists](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/ListIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
